# 📘 부록: Jev 이미지 입력 사용법

Jev 1.14부터 판정 모델 Jev가 그림을 입력으로 받고, 그림 속 위치를 상자로 답하는 Box 질문이 생겼습니다. 이 부록은 앞에서 배운 Noul, Choice, Score와 새 Box를 그림에 쓰는 법을 LangChain Runnable로 짧게 봅니다. 그림은 과일 사진과 그림 여섯 장입니다.

**실행 환경:** CPU만 씁니다. `.env`의 `TYPESAFE_API_KEY`와 `typesafe-sdk` 0.7.5 이상이 필요합니다. 공식 문서: [TypeSafe Images](https://docs.typesafe.ai/images)

**학습 목표**

1. **실험 기능 헤더와 그림 크기 조건을 지켜 Jev에 그림을 보내는 LangChain Runnable을 만듭니다.**
2. **그림에 Noul, Choice, Score, Box 질문을 하고 답을 읽습니다.**
3. **`batch`로 여러 그림을 한 번에 판정합니다.**

## 1. Jev 1.14에서 달라진 것

| 기능 | 쓸 수 있는 모델 | 쓰는 법 |
|---|---|---|
| 그림 입력 | `jev-1.13.0`, `jev-1.14.0` | `Image(경로)`를 `state`나 `criteria`에 넣음 |
| Box 질문 | `jev-1.14.0` | 그림 속 대상의 위, 아래, 왼쪽, 오른쪽 경계를 비율(0~1)로 답함 |

- **실험 기능 헤더:** 그림이 든 요청마다 `TypeSafe-Danger-Zone-Images: true` 헤더를 붙입니다. 없으면 `400 Images are experimental`로 거절됩니다. 공식 문서는 요청 형식이 앞으로 바뀔 수 있다고 밝힙니다.
- **그림 조건:** PNG, JPEG만 받고, 가로와 세로가 각각 280~1568px여야 합니다. 벗어나면 줄여 주지 않고 거절합니다(조건 전체는 2-2).
- **비용:** 그림도 보통 입력 토큰으로 계산하고, 그림이 클수록 토큰이 늘어납니다.
- **LangChain:** 앞 단원에서 쓴 LangChain 통합 패키지 `langchain-typesafe`의 `TypeSafeClassifier`는 아직 그림을 보내지 못합니다. 마지막 판(0.0.1a3)이 Jev 1.14보다 먼저 나왔고, [LangChain 통합 문서](https://docs.langchain.com/oss/python/integrations/providers/typesafe)도 `state`를 문자열, JSON, LangChain 메시지로만 안내하며 이미지와 Box는 다루지 않습니다. 그래서 그림을 지원하는 공식 SDK `typesafe_sdk`로 보내는 함수를 `RunnableLambda`로 감싸, `TypeSafeClassifier`와 같은 입력(`{"state": ..., "questions": ...}`)과 같은 답(`nouls`, `choices`, `scores`)으로 씁니다. 통합 패키지가 그림을 지원하게 되면 이 부분만 바꾸면 됩니다.

## 2. 실습 준비

### 2-1. 그림

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from PIL import Image as PILImage  # typesafe_sdk의 Image와 이름이 겹쳐 PIL 쪽은 PILImage로 부릅니다

material_dir = Path(".")  # 단원 폴더
image_dir = material_dir / "data" / "부록_Jev이미지"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 단원 폴더의 .env를 먼저 읽고, 없는 키만 상위 실습자료 폴더의 .env에서 채웁니다
load_dotenv(material_dir / ".env")
load_dotenv(material_dir.resolve().parent / ".env")



def show_images(paths):
    """그림 여러 장을 한 줄로 보여 줍니다. 제목은 파일 이름과 크기입니다."""
    fig, axes = plt.subplots(1, len(paths), figsize=(3.2 * len(paths), 3.4), squeeze=False)  # squeeze=False: 한 장이어도 axes[0]이 목록
    for ax, path in zip(axes[0], paths):
        picture = PILImage.open(path)
        ax.imshow(picture)
        ax.set_title(f"{Path(path).stem} {picture.size[0]}x{picture.size[1]}")
        ax.axis("off")
    plt.show()


names = ["fruit_basket", "apple_photo", "apple_watercolor", "apple_pencil", "banana", "fruit_market"]
paths = []
for name in names:
    paths.append(image_dir / f"{name}.jpg")
show_images(paths)

### 2-2. 그림 크기 맞추기

Jev가 받는 그림 조건은 아래와 같습니다([TypeSafe Images](https://docs.typesafe.ai/images)). 하나라도 벗어나면 Jev는 크기를 맞춰 주지 않고 요청 전체를 거절합니다.

| 조건 | 범위 | 벗어나면 |
|---|---|---|
| 파일 형식 | PNG, JPEG(움직이는 PNG 제외) | `400` 거절. GIF, WebP, HEIC는 PNG나 JPEG로 바꿔 보냅니다 |
| 가로, 세로 | 각각 280px 이상 1568px 이하 | `400 Invalid image dimensions` |
| 전체 픽셀 수 | 78,400(280 x 280) 이상 2,458,624(1568 x 1568) 이하 | `400` 거절 |
| 파일 크기 | 그림 한 장 16 MiB 이하 | `413` 거절 |
| 요청 전체 | 그림을 모두 합해 32 MiB 이하 | `413` 거절 |

가로, 세로를 280~1568px 안에 두면 전체 픽셀 수 조건도 함께 지켜집니다. 아래 `fit_for_jev`는 긴 변이 1568px를 넘으면 비율을 지켜 줄이고, 짧은 변이 280px보다 작으면 비율을 지켜 키운 뒤 JPEG로 저장합니다. 그래서 파일 크기도 16 MiB보다 훨씬 작아집니다. 다만 가로세로 비율이 5.6:1(1568 / 280)보다 긴 그림은 두 변을 함께 범위에 넣을 수 없으므로 여러 조각으로 잘라 보냅니다. 과일 가게 사진(`fruit_market`)은 휴대폰 사진처럼 3072x2048이라 그대로 보내면 `400 Invalid image dimensions`로 거절되므로, 이 함수로 맞춰 보냅니다.

In [ ]:
def fit_for_jev(path):
    """가로, 세로를 Jev가 받는 280~1568px 안으로 맞춘 그림을 output/에 저장하고 그 경로를 돌려줍니다."""
    picture = PILImage.open(path).convert("RGB")

    # 긴 변이 1568px를 넘으면 비율을 지켜 줄입니다(thumbnail은 줄이기만 합니다)
    picture.thumbnail((1568, 1568))

    # 짧은 변이 280px보다 작으면 비율을 지켜 키웁니다
    short_side = min(picture.size)
    if short_side < 280:
        scale = 280 / short_side
        picture = picture.resize((round(picture.width * scale), round(picture.height * scale)), PILImage.LANCZOS)

    out_path = output_dir / (Path(path).stem + "_jev.jpg")
    picture.save(out_path, quality=90)
    return out_path


market_source = image_dir / "fruit_market.jpg"
market_path = fit_for_jev(market_source)
print("원래 크기:", PILImage.open(market_source).size, "-> 맞춘 크기:", PILImage.open(market_path).size)

### 2-3. 그림을 보내는 Runnable

SDK 호출 하나를 함수로 만들고 `RunnableLambda`로 감쌉니다. Box 질문이 1.14부터라 모델은 `jev-1.14.0`으로 고정합니다. 질문 객체(`Noul`, `Choice`, `Score`, `Box`)는 이름은 앞 단원과 같지만 `typesafe_sdk`에서 가져옵니다(`langchain_typesafe`의 질문 객체는 SDK가 받지 않습니다).

In [ ]:
from langchain_core.runnables import RunnableLambda
from typesafe_sdk import Image, TypeSafeClient

JEV_IMAGE_MODEL = "jev-1.14.0"
IMAGE_HEADERS = {"TypeSafe-Danger-Zone-Images": "true"}  # 그림이 든 요청마다 붙이는 실험 기능 헤더
client = TypeSafeClient()  # TYPESAFE_API_KEY 환경 변수를 읽습니다


def ask_jev(request):
    """{"state": ..., "questions": ...}를 받아 그림이 든 요청을 보내고 Jev의 답을 돌려줍니다."""
    return client.system_one(
        model=JEV_IMAGE_MODEL,
        state=request["state"],  # Image(경로)를 넣으면 SDK가 파일을 올리고 참조로 바꿉니다
        questions=request["questions"],
        extra_headers=IMAGE_HEADERS,
    )


jev_image = RunnableLambda(ask_jev)  # invoke, batch를 쓰는 Runnable

## 3. 그림에 묻기

### 3-1. Noul: 참 또는 거짓

`nouls[이름].noul`은 '참'일 확률입니다. 사진에 사과가 있는지 묻습니다.

In [ ]:
show_images([image_dir / "fruit_basket.jpg", image_dir / "banana.jpg"])

In [ ]:
from typesafe_sdk import Noul

apple_question = Noul(instructions="그림에 사과가 있는가?")

rows = []
for name in ["fruit_basket", "banana"]:
    response = jev_image.invoke({"state": Image(image_dir / f"{name}.jpg"), "questions": {"apple": apple_question}})
    rows.append({"그림": name, "참일 확률": round(response.nouls["apple"].noul, 2), "입력 토큰": response.usage.input_tokens})
display(pd.DataFrame(rows))

### 3-2. Choice: 하나 고르기

`choices[이름].choice`는 고른 선택지, `probabilities`는 선택지마다의 확률입니다. 그림의 화풍을 고르게 합니다.

In [ ]:
show_images([image_dir / "apple_pencil.jpg"])

In [ ]:
from typesafe_sdk import Choice

style_question = Choice(
    instructions="그림의 화풍을 고르세요.",
    criteria={"photo": "사진", "watercolor": "수채화", "pencil": "연필 스케치"},
)

response = jev_image.invoke({"state": Image(image_dir / "apple_pencil.jpg"), "questions": {"style": style_question}})
answer = response.choices["style"]
print("고른 화풍:", answer.choice)
display(pd.Series(answer.probabilities).round(2))

### 3-3. Score: 두 그림 견주기

`state`에 그림 여러 장을 이름 붙인 딕셔너리로 넣고, 질문에서 그 이름을 백틱으로 가리킵니다. `criteria`의 순서가 점수(0점부터)이고, `scores[이름].score`는 단계별 확률로 가중 평균한 값입니다. 사과 사진을 기준으로, 후보 그림이 같은 과일을 그렸는지 점수로 봅니다.

In [ ]:
show_images([image_dir / "apple_photo.jpg", image_dir / "apple_watercolor.jpg", image_dir / "banana.jpg"])  # 첫 장이 기준, 나머지가 후보

In [ ]:
from typesafe_sdk import Score

same_fruit = Score(
    instructions="`후보` 그림의 과일이 `기준` 사진의 과일과 같은 종류인 정도",
    criteria=["전혀 다른 과일", "색만 비슷", "모양 일부가 같음", "거의 같은 종류", "같은 종류"],
)

rows = []
for name in ["apple_watercolor", "banana"]:
    state = {"기준": Image(image_dir / "apple_photo.jpg"), "후보": Image(image_dir / f"{name}.jpg")}  # 질문이 가리킬 이름을 붙여 두 장을 넣습니다
    response = jev_image.invoke({"state": state, "questions": {"same_fruit": same_fruit}})
    rows.append({"후보": name, "점수(0~4)": round(response.scores["same_fruit"].score, 2), "입력 토큰": response.usage.input_tokens})
display(pd.DataFrame(rows))

### 3-4. 글과 그림을 함께 넣기

`state`는 글, 그림, 딕셔너리, 리스트를 모두 받으므로, 딕셔너리 하나에 글과 그림을 함께 넣을 수 있습니다. 주문 내용(글)과 배송 사진(그림)을 넣고, 사진에 주문한 과일이 모두 있는지 묻습니다.

In [ ]:
show_images([image_dir / "fruit_basket.jpg"])  # 배송 사진

In [ ]:
match_question = Noul(instructions="`배송 사진`에 `주문 내용`의 과일이 모두 들어 있는가?")

rows = []
for order in ["사과 3개, 바나나 1송이, 포도 1송이", "사과 3개, 수박 1통"]:
    state = {"주문 내용": order, "배송 사진": Image(image_dir / "fruit_basket.jpg")}  # 글과 그림을 한 딕셔너리에
    response = jev_image.invoke({"state": state, "questions": {"match": match_question}})
    rows.append({"주문 내용": order, "참일 확률": round(response.nouls["match"].noul, 2)})
display(pd.DataFrame(rows))

### 3-5. Box: 위치 찾기

Box는 `instructions`가 가리키는 것을 찾아 `boxes[이름].box`에 상자의 `top`, `bottom`(그림 높이에 대한 비율), `left`, `right`(너비에 대한 비율)를 돌려줍니다. Box에는 `criteria`가 없습니다. 바구니 사진에서 바나나를 찾습니다.

In [ ]:
show_images([image_dir / "fruit_basket.jpg"])

In [ ]:
from PIL import ImageDraw
from typesafe_sdk import Box

sample_path = image_dir / "fruit_basket.jpg"
response = jev_image.invoke({"state": Image(sample_path), "questions": {"banana": Box(instructions="바나나 송이")}})
answer = response.boxes["banana"]
print("상자:", answer.box, "| confidence:", answer.confidence)

# 비율 * 그림 크기 = 픽셀 위치
picture = PILImage.open(sample_path).convert("RGB")
width, height = picture.size
ImageDraw.Draw(picture).rectangle(
    [answer.box.left * width, answer.box.top * height, answer.box.right * width, answer.box.bottom * height],  # [왼쪽, 위, 오른쪽, 아래]
    outline="red",
    width=6,
)
plt.figure(figsize=(4, 4))
plt.imshow(picture)
plt.axis("off")
plt.show()

### 3-6. Choice의 기준에 그림 넣기

그림은 `state`뿐 아니라 `criteria`에도 넣을 수 있습니다. 글로 설명하기 어려운 기준을 그림으로 보여 주는 방식입니다. 가게의 상품 사진 두 장(사과, 바나나)을 선택지의 기준으로 두고, 들어온 그림이 어느 상품인지 고르게 합니다. 상품에 없는 과일은 `none`을 고르게 합니다.

In [ ]:
show_images([image_dir / "apple_photo.jpg", image_dir / "banana.jpg"])  # 선택지 기준: 상품 사진

In [ ]:
show_images([image_dir / "apple_watercolor.jpg", image_dir / "apple_pencil.jpg", image_dir / "grapes.jpg"])  # 고를 그림

In [ ]:
product_question = Choice(
    instructions="그림 속 과일이 어느 상품과 같은 과일인지 고르세요.",
    criteria={
        "apple": Image(image_dir / "apple_photo.jpg"),  # 선택지 설명 대신 상품 사진
        "banana": Image(image_dir / "banana.jpg"),
        "none": "위 두 상품 어느 것과도 다른 과일",
    },
)

rows = []
for name in ["apple_watercolor", "apple_pencil", "grapes"]:
    response = jev_image.invoke({"state": Image(image_dir / f"{name}.jpg"), "questions": {"product": product_question}})
    answer = response.choices["product"]
    rows.append({"그림": name, "고른 상품": answer.choice, "확률": round(answer.probabilities[answer.choice], 2)})
display(pd.DataFrame(rows))

## 4. 여러 그림 한 번에 판정

`batch`는 입력 리스트를 받아 같은 순서로 답을 돌려주고, `max_concurrency`로 동시에 보낼 요청 수를 정합니다. 2-2에서 크기를 맞춘 과일 가게 사진까지 여섯 장에 화풍 질문과 사과 질문을 함께 묻습니다. 질문 묶음에 여러 질문을 넣으면 한 번의 요청으로 모두 답합니다.

In [ ]:
batch_paths = []
for name in ["fruit_basket", "apple_photo", "apple_watercolor", "apple_pencil", "banana"]:
    batch_paths.append(image_dir / f"{name}.jpg")
batch_paths.append(market_path)  # 2-2에서 1568x1045로 맞춘 사진

show_images(batch_paths)

In [ ]:
# 그림마다 묻는 질문 두 개: 화풍 고르기(Choice)와 사과가 있는지(Noul)
batch_questions = {
    "style": Choice(
        instructions="그림의 화풍을 고르세요.",
        criteria={"photo": "사진", "watercolor": "수채화", "pencil": "연필 스케치"},
    ),
    "apple": Noul(instructions="그림에 사과가 있는가?"),
}

requests = []
for path in batch_paths:
    requests.append({"state": Image(path), "questions": batch_questions})

responses = jev_image.batch(requests, config={"max_concurrency": 6})  # 동시에 보내는 요청 수

rows = []
for path, response in zip(batch_paths, responses):
    style = response.choices["style"]
    rows.append({"그림": path.stem, "화풍": style.choice, "화풍 확률": round(style.probabilities[style.choice], 2), "사과 확률": round(response.nouls["apple"].noul, 2)})
display(pd.DataFrame(rows))

## 5. 핵심 정리

- **보내기:** SDK 호출을 `RunnableLambda`로 감싼 `jev_image`에 `{"state": Image(경로), "questions": {...}}`를 넣어 `invoke`, `batch`로 부릅니다. 요청마다 `TypeSafe-Danger-Zone-Images: true` 헤더가 붙고, 가로, 세로가 280~1568px를 벗어난 그림은 `fit_for_jev`로 맞춰 보냅니다.
- **질문과 답:** Noul은 `nouls`(참일 확률), Choice는 `choices`(고른 선택지와 확률), Score는 `scores`(단계의 가중 평균), Box는 `boxes`(그림 크기에 대한 비율로 된 상자)로 읽습니다. `state`에는 글과 그림을 한 딕셔너리에 함께 넣을 수 있고, 그림은 `criteria`에도 넣을 수 있습니다.
- **주의:** 실험 기능이라 형식이 바뀔 수 있고, 같은 그림, 같은 질문도 실행마다 확률이 조금씩 다릅니다. Box 위치는 대략적입니다. EXIF 회전은 적용하지 않으므로 그림을 똑바로 세워 보냅니다.